# NB_01  Bronze Ingestion

Raw ingestion of all source files into the Lakehouse Bronze layer.

Bronze is an exact mirror of the source. No transformations are applied.
Each file lands as a Delta table. If reprocessing is needed, this is where we start.

Sources:
- CSV: Orders, Order_Details, Customers, Products, Categories, Divisions, Shippers
- CSV (multi-file): Shipments (3 files merged into 1 table)
- XML: Suppliers
- Excel: Employees + Offices (separate sheets), Budget

Design decisions:
- Schema inferred at read time, no manual DDL
- Column names sanitized for Delta compatibility (spaces and special chars replaced)
- _ingested_at timestamp appended to every table for lineage tracking
- SummaryReport.xls explicitly excluded  validation artifact, not a data source

In [ ]:
# Importing Libs 
# Treat chars 
from pyspark.sql.functions import current_timestamp, input_file_name, lit
from pyspark.sql.types import *
import pandas as pd, re, xml.etree.ElementTree as ET

BASE_PATH = "Files/bronze"

def clean_cols(df):
    # Replace spaces and special chars in column names to keep Delta compatibility
    new_cols = [re.sub(r'[ ,;{}()\n\t=]', '_', c).strip('_') for c in df.columns]
    for old, new in zip(df.columns, new_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)
    return df

print(f"Bronze ingestion started | base path: {BASE_PATH}")
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
# Plain CSV files > one subfolder per source, one Delta table per file
# v2 FIX: encoding=windows-1252 (correct codec for Windows-1252 source files)
# v2 FIX: clean_cols() applied to all CSV tables for Delta column name compatibility

def ingest_csv(subfolder, table_name):
    df = (spark.read
            .option("header", True)
            .option("inferSchema", True)
            .option("encoding", "windows-1252")
            .csv(f"{BASE_PATH}/{subfolder}/"))
    df = clean_cols(df)
    df = df.withColumn("_ingested_at", current_timestamp())
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(f"bronze_{table_name}")
    print(f"  bronze_{table_name}: {df.count()} rows | {len(df.columns)} cols")

ingest_csv("orders",        "orders")
ingest_csv("order_details", "order_details")
ingest_csv("customers",     "customers")
ingest_csv("products",      "products")
ingest_csv("categories",    "categories")
ingest_csv("divisions",     "divisions")
ingest_csv("shippers",      "shippers")

In [ ]:
# Shipments: 3 partitioned CSV files merged into a single table
# Keeping the _source_file column tracks which file each row came from
# v2 FIX: encoding=windows-1252
# v2 FIX: clean_cols() applied for Delta column name compatibility

df_ship = (spark.read
             .option("header", True)
             .option("inferSchema", True)
             .option("encoding", "windows-1252")
             .csv(f"{BASE_PATH}/shipments/"))

df_ship = clean_cols(df_ship)
df_ship = (df_ship
             .withColumn("_source_file", input_file_name())
             .withColumn("_ingested_at", current_timestamp()))

df_ship.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("bronze_shipments")
print(f"bronze_shipments: {df_ship.count()} rows")
df_ship.groupBy("_source_file").count().show(truncate=False)

In [ ]:
# Suppliers — XML file parsed with ElementTree
# Each child tag of the root element becomes a column
# v2 FIX: clean_cols() applied for Delta column name compatibility

xml_path = "/lakehouse/default/Files/bronze/suppliers/Suppliers.xml"
with open(xml_path, "r", encoding="utf-8") as f:
    xml_content = f.read()

root = ET.fromstring(xml_content)
rows = [{child.tag: child.text for child in supplier} for supplier in root]

df_sup = spark.createDataFrame(rows)
df_sup = clean_cols(df_sup)
df_sup = df_sup.withColumn("_ingested_at", current_timestamp())
df_sup.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("bronze_suppliers")
print(f"bronze_suppliers: {df_sup.count()} rows")
df_sup.show(3)

In [ ]:
# Budget — Excel with a title row on row 1 and real headers on row 2
# header=1 skips the "QWT Budget" title row
# Kept as strings — all type casting happens in Silver
# v2 FIX: clean_cols() applied for Delta column name compatibility

pdf = pd.read_excel("/lakehouse/default/Files/bronze/budget/Budget.xlsx", header=1)
pdf = pdf.dropna(how='all')  # drop fully empty rows (Excel artifact)

df_budget = spark.createDataFrame(pdf.astype(str))
df_budget = clean_cols(df_budget)
df_budget = df_budget.withColumn("_ingested_at", current_timestamp())
df_budget.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("bronze_budget")
print(f"bronze_budget: {df_budget.count()} rows")
df_budget.show()

In [ ]:
# The Data for Employees and Offices -> two sheets from the same Excel file
# Need to clean and format the Column names with spaces are sanitized via clean_cols()

emp_path = "/lakehouse/default/Files/bronze/employees_offices/Employees Offices.xlsx"

pdf_emp = pd.read_excel(emp_path, sheet_name="Employee")
pdf_off = pd.read_excel(emp_path, sheet_name="Office")

df_emp = clean_cols(spark.createDataFrame(pdf_emp.astype(str))).withColumn("_ingested_at", current_timestamp())
df_off = clean_cols(spark.createDataFrame(pdf_off.astype(str))).withColumn("_ingested_at", current_timestamp())

df_emp.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("bronze_employees")
df_off.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("bronze_offices")

print(f"bronze_employees: {df_emp.count()} rows | cols: {df_emp.columns}")
print(f"bronze_offices:   {df_off.count()} rows | cols: {df_off.columns}")

In [ ]:
# Final validation: confirm all 12 Bronze tables were created
tables = spark.sql("SHOW TABLES").filter("tableName LIKE 'bronze_%'")
tables.select("tableName").orderBy("tableName").show(20, truncate=False)
print(f"Total bronze tables: {tables.count()}")